# Derin Öğrenme ile Nesne Tespiti: YOLO ve SSD (Gerçek Veri Setleriyle)

Bu not defteri, tek aşamalı (single-stage) nesne tespit mimarilerinin iki
önemli temsilcisini **kavramsal açıklama + satır satır yorumlanmış kod**
formatında, **gerçek veri setleri** kullanarak ele alır:

1. **YOLO (You Only Look Once)** — `ultralytics` kütüphanesi ile **COCO128**
   (gerçek COCO fotoğraflarından oluşan küçük bir alt küme) üzerinde eğitim ve çıkarım
2. **SSD (Single Shot MultiBox Detector)** — `torchvision` ile **Pascal VOC 2007**
   (gerçek, etiketlenmiş fotoğraflardan oluşan klasik nesne tespiti veri seti) üzerinde
   ince ayar (fine-tuning) ve çıkarım

> **Önemli not:** Bu iki bölüm, önceki CNN/U-Net/Mask R-CNN defterinden farklı
> olarak **sahte (rastgele) veri değil, gerçek indirilen veri setleri**
> kullanır. Kod hücreleri veri setini otomatik olarak internetten indirir.
> Kısıtlı/izole ağ ortamlarında (örn. kurumsal sandbox) indirme adımları
> başarısız olabilir; bu durumda aynı kodu internet erişimi olan normal bir
> ortamda (kendi bilgisayarınız, Google Colab, Kaggle Notebooks vb.)
> çalıştırmanız yeterlidir.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")


---
## 1. YOLO (You Only Look Once)

### Konu Anlatımı

YOLO, görüntüyü **tek bir geçişte (single pass)** işleyerek hem nesnelerin
konumunu hem de sınıfını aynı anda tahmin eden bir mimaridir. Faster R-CNN
gibi iki aşamalı (önce bölge öner, sonra sınıflandır) yöntemlerin aksine,
YOLO tespit problemini doğrudan bir **regresyon problemi** olarak ele alır;
bu da onu çok hızlı yapar (gerçek zamanlı uygulamalara uygundur).

Temel çalışma mantığı:

1. **Izgara (Grid) Bölme:** Görüntü, S x S boyutunda bir ızgaraya bölünür.
   Her ızgara hücresi, merkezi kendi içine düşen nesnelerden sorumludur.

2. **Tahmin Vektörü:** Her hücre; sınırlayıcı kutu koordinatları (x, y, w, h),
   bir **nesnellik (objectness) skoru** (o hücrede nesne olma olasılığı) ve
   sınıf olasılıklarını aynı anda tahmin eder.

3. **Çapa Kutuları / Ankor-Free Kafa:** Eski YOLO sürümleri önceden
   tanımlanmış çapa (anchor) kutuları kullanırken, YOLOv8 gibi güncel
   sürümler **ankor-free (anchor-free)** bir yaklaşım kullanır; bu, kutu
   tahminini basitleştirir ve doğruluğu artırır.

4. **Non-Maximum Suppression (NMS):** Aynı nesne için üretilen çakışan
   çoklu kutulardan en yüksek güven skoruna sahip olanı tutup diğerlerini
   eler.

**Tipik kullanım alanı:** Gerçek zamanlı nesne tespiti (güvenlik kameraları,
otonom araçlar, canlı video analizleri).

Bu bölümde, sıfırdan mimari kurmak yerine (üretim ortamlarında da yaygın
yaklaşım budur) `ultralytics` kütüphanesinin **YOLOv8** modelini kullanıyoruz
ve **COCO128** veri setiyle ince ayar (fine-tuning) yapıyoruz. COCO128,
gerçek COCO 2017 veri setinden alınan **128 gerçek fotoğraf ve gerçek
insan etiketli kutu/sınıf bilgisinden** oluşan, hızlı denemeler için
tasarlanmış küçük ama tamamen gerçek bir alt kümedir.


In [ ]:
# ultralytics kütüphanesini kuruyoruz: YOLOv8 modelini, eğitim döngüsünü ve
# veri seti indirme mantığını hazır olarak sağlar
!pip install ultralytics --quiet


In [ ]:
from ultralytics import YOLO

# "yolov8n.pt": COCO veri setinde önceden eğitilmiş en küçük (nano) YOLOv8 modeli
# Bu satır çalıştırıldığında ağırlık dosyası otomatik olarak internetten indirilir
model_yolo = YOLO("yolov8n.pt")

print(model_yolo.model)  # Modelin katman yapısını (backbone + head) yazdırır


In [ ]:
# --- Gerçek Veri Setiyle İnce Ayar (Fine-tuning) ---
# "coco128.yaml": ultralytics'in sağladığı hazır yapılandırma dosyası.
# Bu dosya çalıştırıldığında COCO128 veri seti (128 gerçek COCO fotoğrafı +
# gerçek nesne kutuları/sınıfları) otomatik olarak indirilir ve
# data/coco128/ klasörüne yerleştirilir.
#
# epochs=3   : Hızlı bir demo için düşük tutuldu (gerçek projede 50-300 arası olur)
# imgsz=640  : Görüntülerin yeniden boyutlandırılacağı kare boyut (piksel)
# batch=8    : Her adımda işlenecek görüntü sayısı

sonuclar = model_yolo.train(
    data="coco128.yaml",
    epochs=3,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
)


In [ ]:
# --- Doğrulama (Validation) ---
# Eğitilen modelin performansını, veri setinin ayrılmış doğrulama bölümünde ölçer
# mAP50: IoU eşiği 0.5'te ortalama kesinlik (mean Average Precision) — nesne
# tespitinde standart başarı metriği
metrikler = model_yolo.val()
print(f"mAP50    : {metrikler.box.map50:.4f}")
print(f"mAP50-95 : {metrikler.box.map:.4f}")


In [ ]:
# --- Gerçek Bir Görüntü Üzerinde Çıkarım (Inference) ---
from ultralytics.utils import ASSETS  # ultralytics ile birlikte gelen örnek gerçek fotoğraflar

ornek_gorsel = ASSETS / "bus.jpg"  # Gerçek bir sokak fotoğrafı (otobüs + yayalar)

# predict(): görüntüyü modele verir, tespit edilen kutuları/sınıfları/skorları döner
# conf=0.25: sadece %25'ten yüksek güven skoruna sahip tespitleri göster
tahmin_sonuclari = model_yolo.predict(source=ornek_gorsel, conf=0.25, save=True)

# Sonuçları görselleştirme: kutular ve etiketler görüntü üzerine otomatik çizilir
sonuc_gorseli = tahmin_sonuclari[0].plot()  # BGR formatında numpy dizisi döner

plt.figure(figsize=(10, 7))
plt.imshow(sonuc_gorseli[..., ::-1])  # BGR -> RGB dönüşümü (matplotlib RGB bekler)
plt.axis("off")
plt.title("YOLOv8 Tespit Sonucu (Gerçek Fotoğraf)")
plt.show()

# Tespit edilen her nesnenin sınıf adı ve güven skorunu yazdır
for kutu in tahmin_sonuclari[0].boxes:
    sinif_id = int(kutu.cls[0])
    sinif_adi = model_yolo.names[sinif_id]
    skor = float(kutu.conf[0])
    print(f"Nesne: {sinif_adi:<15} Güven Skoru: {skor:.2f}")


---
## 2. SSD (Single Shot MultiBox Detector)

### Konu Anlatımı

SSD de YOLO gibi tek aşamalı bir tespit mimarisidir, fakat kutu tahminini
farklı bir stratejiyle yapar:

1. **Çok Ölçekli Özellik Haritaları:** SSD, omurga ağının (backbone, örn.
   VGG16) **farklı derinliklerindeki** özellik haritalarını kullanır. Sığ
   katmanlar küçük nesneleri, derin katmanlar büyük nesneleri yakalamada
   daha iyidir. Böylece tek bir çözünürlük yerine **çok ölçekli** tespit
   yapılır.

2. **Varsayılan Kutular (Default / Anchor Boxes):** Her özellik haritasındaki
   her konumda, farklı en-boy oranlarına (aspect ratio) sahip önceden
   tanımlanmış "varsayılan kutular" bulunur. Model, bu kutuların gerçek
   nesne kutusuna göre ofsetini (kaydırma miktarını) ve sınıf olasılığını
   tahmin eder.

3. **Eşleştirme (Matching) Stratejisi:** Eğitim sırasında, her gerçek (ground
   truth) nesne kutusu, kendisiyle en yüksek IoU (kesişim/birleşim oranı)
   değerine sahip varsayılan kutu(lar) ile eşleştirilir; bu kutular "pozitif"
   örnek olarak etiketlenir.

4. **Zor Negatif Madenciliği (Hard Negative Mining):** Arkaplan (nesne
   olmayan) kutuları pozitif kutulardan çok daha fazla olduğu için, eğitim
   dengesizliğini önlemek amacıyla sadece en yüksek kayba sahip negatif
   örnekler kullanılır.

**Tipik kullanım alanı:** Mobil/gömülü cihazlarda hız-doğruluk dengesi
gerektiren tespit görevleri.

Bu bölümde, `torchvision`'ın **SSD300 + VGG16** mimarisini, gerçek ve klasik
bir nesne tespiti veri seti olan **Pascal VOC 2007** üzerinde ince ayar
yaparak kullanıyoruz. VOC 2007, 20 farklı nesne sınıfı (insan, araba, kedi,
bisiklet vb.) için gerçek fotoğraflar ve insan tarafından çizilmiş
sınırlayıcı kutu etiketleri içerir.


In [ ]:
import torchvision
from torchvision.datasets import VOCDetection
from torchvision import transforms as T
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.vgg import VGG16_Weights

# Pascal VOC 2007'deki 20 nesne sınıfı + "arkaplan" (index 0, zorunlu bir sınıf)
VOC_SINIFLARI = [
    "arkaplan", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus",
    "car", "cat", "chair", "cow", "diningtable", "dog", "horse",
    "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
sinif_to_id = {isim: i for i, isim in enumerate(VOC_SINIFLARI)}
NUM_CLASSES = len(VOC_SINIFLARI)  # 21 (20 gerçek sınıf + arkaplan)

print(f"Toplam sınıf sayısı (arkaplan dahil): {NUM_CLASSES}")


In [ ]:
class VOCTespitVeriSeti(torch.utils.data.Dataset):
    """
    torchvision'ın ham VOCDetection çıktısını (XML tabanlı sözlük), Mask R-CNN'de
    de kullandığımız standart nesne tespiti formatına (boxes + labels tensörleri)
    dönüştüren sarmalayıcı (wrapper) sınıf.
    """
    def __init__(self, root="./data", year="2007", image_set="train", indir=True):
        # download=True: veri seti diskte yoksa gerçek VOC 2007 arşivini indirir
        # (görüntüler + XML etiket dosyaları içeren gerçek, etiketli veri)
        self.voc = VOCDetection(root=root, year=year, image_set=image_set, download=indir)
        self.to_tensor = T.ToTensor()  # PIL görüntüyü [0,1] aralığında tensöre çevirir

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        img, annotation = self.voc[idx]  # Ham PIL görüntü + XML'den ayrıştırılmış sözlük
        img = self.to_tensor(img)

        nesneler = annotation["annotation"]["object"]
        # Görüntüde tek nesne varsa VOC bunu dict olarak döner, listeye çeviriyoruz
        if isinstance(nesneler, dict):
            nesneler = [nesneler]

        kutular, etiketler = [], []
        for nesne in nesneler:
            bbox = nesne["bndbox"]
            xmin, ymin = float(bbox["xmin"]), float(bbox["ymin"])
            xmax, ymax = float(bbox["xmax"]), float(bbox["ymax"])
            kutular.append([xmin, ymin, xmax, ymax])
            etiketler.append(sinif_to_id[nesne["name"]])

        hedef = {
            "boxes": torch.tensor(kutular, dtype=torch.float32),
            "labels": torch.tensor(etiketler, dtype=torch.int64),
        }
        return img, hedef


In [ ]:
# --- Gerçek Veri Setini Yükleme ---
# NOT: Bu satır çalıştırıldığında Pascal VOC 2007 arşivi (~460 MB) gerçekten
# internetten indirilir. İnternet erişimi kısıtlı ortamlarda bu adım
# başarısız olabilir; tam internet erişimi olan bir ortamda çalıştırın.

egitim_veri_seti = VOCTespitVeriSeti(root="./data", year="2007", image_set="train", indir=True)
print(f"Eğitim veri setindeki gerçek görüntü sayısı: {len(egitim_veri_seti)}")

# Farklı görüntülerin farklı sayıda nesne içerebilmesi nedeniyle, PyTorch'un
# varsayılan collate fonksiyonu yerine kutuları/etiketleri liste olarak tutan
# özel bir collate fonksiyonu tanımlıyoruz
def collate_fn(batch):
    return tuple(zip(*batch))

egitim_yukleyici = torch.utils.data.DataLoader(
    egitim_veri_seti,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
)


In [ ]:
# --- Model Kurulumu: Pretrained Backbone + Yeni Tespit Başlığı ---
# weights=None            : COCO üzerinde eğitilmiş hazır tespit başlığını kullanmıyoruz
#                            (çünkü sınıf sayımız VOC'a göre farklı: 21)
# weights_backbone=DEFAULT: Omurga ağı (VGG16) ImageNet üzerinde önceden eğitilmiş
#                            ağırlıklarla başlar (transfer learning) — sıfırdan
#                            öğrenmek yerine genel görsel özelliklerden faydalanılır
# num_classes=21           : Sınıflandırma başlığı otomatik olarak bu sayıya göre kurulur
model_ssd = ssd300_vgg16(
    weights=None,
    weights_backbone=VGG16_Weights.DEFAULT,
    num_classes=NUM_CLASSES,
).to(device)

print(f"Toplam parametre sayısı: {sum(p.numel() for p in model_ssd.parameters()):,}")


In [ ]:
# --- Gerçek Veri ile İnce Ayar (Fine-tuning) Döngüsü ---
optimizer = torch.optim.SGD(model_ssd.parameters(), lr=0.005, momentum=0.9, weight_decay=5e-4)

EPOCHS = 2  # Hızlı bir demo için düşük tutuldu (gerçek projede çok daha fazla olur)

model_ssd.train()  # Modeli eğitim moduna al
for epoch in range(EPOCHS):
    toplam_kayip = 0.0
    for goruntuler, hedefler in egitim_yukleyici:
        goruntuler = [g.to(device) for g in goruntuler]                              # Her görüntüyü cihaza taşı
        hedefler = [{k: v.to(device) for k, v in h.items()} for h in hedefler]        # Her hedef sözlüğünü cihaza taşı

        # Detection modelleri eğitim modunda görüntü + hedef verildiğinde
        # otomatik olarak kayıp sözlüğü (loss_dict) döner: sınıflandırma + kutu regresyon kaybı
        kayip_sozlugu = model_ssd(goruntuler, hedefler)
        kayip = sum(v for v in kayip_sozlugu.values())  # Tüm kayıp bileşenlerini topla

        optimizer.zero_grad()  # Önceki gradyanları sıfırla
        kayip.backward()       # Geri yayılım
        optimizer.step()       # Ağırlıkları güncelle

        toplam_kayip += kayip.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Ortalama Kayıp: {toplam_kayip/len(egitim_yukleyici):.4f}")


In [ ]:
# --- Gerçek Bir Görüntü Üzerinde Çıkarım (Inference) ---
model_ssd.eval()  # Değerlendirme moduna al

# Eğitim veri setinden gerçek bir örnek görüntü alıyoruz
test_goruntu, _ = egitim_veri_seti[0]

with torch.no_grad():  # Çıkarım sırasında gradyan hesaplama gereksiz, bellekten tasarruf için kapatılır
    tahmin = model_ssd([test_goruntu.to(device)])[0]

# Sadece güven skoru 0.5'ten yüksek olan tespitleri filtrele (düşük güvenli gürültüyü ele)
esik = 0.5
gecerli_indeksler = tahmin["scores"] > esik

import matplotlib.patches as patches

plt.figure(figsize=(9, 7))
plt.imshow(test_goruntu.permute(1, 2, 0).cpu())  # Tensör (C,H,W) -> görüntü (H,W,C) formatına çevir
eksen = plt.gca()

for kutu, etiket, skor in zip(
    tahmin["boxes"][gecerli_indeksler],
    tahmin["labels"][gecerli_indeksler],
    tahmin["scores"][gecerli_indeksler],
):
    xmin, ymin, xmax, ymax = kutu.cpu().numpy()
    dikdortgen = patches.Rectangle(
        (xmin, ymin), xmax - xmin, ymax - ymin,
        linewidth=2, edgecolor="red", facecolor="none"
    )
    eksen.add_patch(dikdortgen)
    plt.text(
        xmin, ymin - 5,
        f"{VOC_SINIFLARI[etiket]} ({skor:.2f})",
        color="white", backgroundcolor="red", fontsize=9,
    )

plt.axis("off")
plt.title("SSD300 Tespit Sonucu (Gerçek VOC Fotoğrafı)")
plt.show()


---
## Özet Karşılaştırma

| Özellik | YOLO | SSD | (Karşılaştırma için) Mask R-CNN |
|---|---|---|---|
| **Aşama sayısı** | Tek aşama | Tek aşama | İki aşama (RPN + Head) |
| **Hız** | Çok yüksek | Yüksek | Daha düşük |
| **Doğruluk (özellikle küçük nesneler)** | İyi (yeni sürümlerde çok iyi) | Orta-iyi | En yüksek |
| **Çıktı** | Kutu + sınıf | Kutu + sınıf | Kutu + sınıf + piksel maskesi |
| **Bu defterde kullanılan gerçek veri seti** | COCO128 (gerçek COCO alt kümesi) | Pascal VOC 2007 | — |
| **Tipik kullanım** | Gerçek zamanlı tespit | Hız/doğruluk dengesi gereken tespit | Yüksek doğruluklu örnek segmentasyonu |

### Sonraki Adımlar
- `epochs` ve `batch` değerlerini artırarak gerçek anlamda performanslı bir model eğitebilirsiniz (GPU önerilir).
- Kendi özel veri setinizi YOLO formatına (`images/`, `labels/*.txt`) veya VOC XML formatına dönüştürerek bu kodları doğrudan kendi projenizde kullanabilirsiniz.
- Eğitilen modelleri kaydetmek için: `model_yolo.save("yolov8_egitilmis.pt")` veya `torch.save(model_ssd.state_dict(), "ssd_egitilmis.pth")`.
- Daha büyük/gerçekçi bir karşılaştırma için tam COCO 2017 veya VOC 2012 veri setleri kullanılabilir (indirme boyutları GB seviyesindedir).
